# 🎯 AI Job Recommendation System
**DecodeLabs Internship 2026 — Project 3**

**Author:** Umar Saeed Jan

---

## What is a Content-Based Recommendation System?
A recommendation system that suggests items based on the **similarity between user preferences and item attributes** — without needing other users' data.

## How This System Works
1. Load job descriptions dataset (1.6M rows, 376 unique roles)
2. Clean and group skills by job role
3. Convert skills to TF-IDF vectors (numerical representation)
4. Take user's skills as input
5. Calculate Cosine Similarity between user vector and all job role vectors
6. Return Top-N most similar job roles

---

## Step 1: Import Libraries & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load the raw Kaggle job descriptions dataset
# Dataset: 1.6 million job postings with 23 columns
df = pd.read_csv('job_descriptions.csv')

print("Dataset Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))

## Step 2: Data Cleaning
We only need two columns: `Role` (job title) and `skills` (required skills).
We also check how many unique job roles exist in the dataset.

In [ ]:
# Keep only the two columns we need for the recommendation engine
df_clean = df[['Role', 'skills']].copy()

# Remove rows where Role or skills is missing
df_clean.dropna(inplace=True)

print("Total rows after cleaning:", len(df_clean))
print("Unique job roles:", df_clean['Role'].nunique())
print("\nTop 20 most frequent roles:")
print(df_clean['Role'].value_counts().head(20))

## Step 3: Group Skills by Job Role
Each role appears thousands of times in the dataset. We combine all skill mentions for each role into one single text — this becomes the "skill profile" for that role.

> Think of it as: *combine all CVs for Data Scientists into one mega-CV*

In [ ]:
# Group by Role and combine all skills into a single string per role
# This gives us one comprehensive skill profile per job role
df_grouped = df_clean.groupby('Role')['skills'].apply(
    lambda x: ' '.join(x)
).reset_index()

# Rename columns for clarity
df_grouped.columns = ['Job_Role', 'Skills']

print("Grouped Dataset Shape:", df_grouped.shape)
print("\nSample (first 5 roles):")
print(df_grouped.head(5))

# Save the cleaned grouped dataset for reuse
df_grouped.to_csv('kaggle_skills_clean.csv', index=False)
print("\nClean dataset saved as: kaggle_skills_clean.csv")

## Step 4: TF-IDF Vectorization
Machines cannot understand words like "Python" or "SQL" — they understand numbers.

**TF-IDF** (Term Frequency - Inverse Document Frequency) converts skill text into numerical vectors:
- **TF** → How often a skill appears in a role's profile (higher = more important for that role)
- **IDF** → How rare a skill is across ALL roles (rarer skill = higher weight)
- **Result** → Unique/specific skills get HIGH weights, generic/common skills get LOW weights

In [ ]:
# Load the clean grouped dataset
df2 = pd.read_csv('kaggle_skills_clean.csv')

# Initialize TF-IDF Vectorizer with tuned parameters
tfidf = TfidfVectorizer(
    analyzer='word',        # Work on word level (not character level)
    ngram_range=(1, 2),     # Consider single words AND two-word phrases (e.g., "machine learning")
    min_df=2,               # Ignore terms appearing in less than 2 roles (too rare to be meaningful)
    max_df=0.85,            # Ignore terms appearing in more than 85% of roles (too generic)
    stop_words='english'    # Remove common English words: "the", "and", "is", etc.
)

# Fit TF-IDF on all job role skill profiles
# Result: a matrix of shape (376 roles x 1327 unique skill terms)
tfidf_matrix = tfidf.fit_transform(df2['Skills'])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)
print("Number of job roles:", tfidf_matrix.shape[0])
print("Number of unique skill terms:", tfidf_matrix.shape[1])
print("\nSample skill terms in vocabulary:")
print(tfidf.get_feature_names_out()[:30])

## Step 5: Recommendation Engine with Cosine Similarity
**Cosine Similarity** measures the angle between two vectors:
- **Score = 1.0** → Perfect match (same direction/interests)
- **Score = 0.0** → No overlap (completely different)

We compare the user's skill vector against ALL 376 job role vectors and return the top matches.

### Pipeline:
```
User Skills → TF-IDF Vector → Cosine Similarity → Sort → Top-N Roles
```

In [ ]:
# ── RECOMMENDATION FUNCTION ──────────────────────────────────────────
def recommend_jobs(user_skills, top_n=5):
    """
    Recommends the top N matching job roles based on user's skills.
    
    Args:
        user_skills (list): List of skills the user knows
        top_n (int): Number of job recommendations to return
    
    Returns:
        DataFrame: Ranked list of matching job roles with similarity scores
    """
    
    # Step 1: Convert user skill list into a single string
    user_input = ' '.join(user_skills)
    
    # Step 2: Transform user input using the SAME TF-IDF vectorizer
    # (Must use .transform(), not .fit_transform() — vocabulary must stay the same)
    user_vector = tfidf.transform([user_input])
    
    # Step 3: Calculate cosine similarity between user vector and ALL job role vectors
    similarity_scores = cosine_similarity(user_vector, tfidf_matrix)
    
    # Step 4: Flatten scores array and sort indices by highest score (descending)
    scores = similarity_scores.flatten()
    top_indices = np.argsort(scores)[::-1][:top_n]
    
    # Step 5: Build and return results DataFrame
    # NOTE: Use df2['Job_Role'] (the grouped clean dataset) — not the raw df
    results = pd.DataFrame({
        'Rank': range(1, top_n + 1),
        'Job Role': df2['Job_Role'].iloc[top_indices].values,
        'Match Score': np.round(scores[top_indices], 4)
    })
    return results


# ── LIVE USER INPUT ───────────────────────────────────────────────────
print("=" * 50)
print("   AI JOB RECOMMENDATION SYSTEM - DecodeLabs")
print("=" * 50)

# Collect skills from user (minimum 3 required for accurate matching)
user_skills = []

print("\nEnter your skills one by one (minimum 3 required)")
print("Type 'done' when finished\n")

while True:
    skill = input(f"Enter skill {len(user_skills) + 1}: ").strip()
    
    if skill.lower() == 'done':
        # Enforce minimum 3 skills for sufficient data density
        if len(user_skills) < 3:
            print(f"Please enter at least 3 skills! You entered {len(user_skills)} so far.\n")
        else:
            break
    elif skill == '':
        print("Skill cannot be empty, please try again.\n")
    else:
        user_skills.append(skill)
        print(f"  Added: {skill}\n")

# Ask how many recommendations the user wants
while True:
    try:
        top_n = int(input("\nHow many job recommendations do you want? (1-10): "))
        if 1 <= top_n <= 10:
            break
        else:
            print("Please enter a number between 1 and 10.")
    except ValueError:
        print("Invalid input, please enter a number.")

# Display final results
print(f"\nYour Skills: {user_skills}")
print("=" * 50)
print("       TOP JOB RECOMMENDATIONS FOR YOU")
print("=" * 50)

results = recommend_jobs(user_skills, top_n=top_n)
print(results.to_string(index=False))

print("\n" + "=" * 50)
print("Powered by DecodeLabs | AI Recommendation System")
print("=" * 50)

---
## Summary

| Component | Details |
|---|---|
| Dataset | Kaggle Job Descriptions (1.6M rows, 376 unique roles) |
| Approach | Content-Based Filtering |
| Feature Extraction | TF-IDF Vectorization (1327 skill terms) |
| Similarity Metric | Cosine Similarity |
| Output | Top-N ranked job role recommendations |

**Key Insight:** TF-IDF rewards specific/rare skills and penalizes generic/common ones — making the matching more meaningful.

---
*DecodeLabs Internship 2026 | Umar Saeed Jan*